# Latent Dirichlet Allocation (LDA)

Latent Dirichlet Allocation (LDA) es una técnica de modelado de temas que se utiliza para descubrir temas ocultos en un conjunto de documentos. Aquí tienes una explicación didáctica:

### Conceptos Clave

1. **Documentos y Palabras**: Un documento es cualquier texto, como un artículo, un libro o una reseña. Cada documento está compuesto por palabras.
2. **Temas**: Un tema es una colección de palabras que suelen aparecer juntas. Por ejemplo, un tema sobre deportes podría incluir palabras como "fútbol", "baloncesto" y "equipo".
3. **Distribución de Temas**: Cada documento es una mezcla de varios temas en diferentes proporciones. Por ejemplo, un artículo podría ser 70% sobre deportes y 30% sobre política.

### ¿Cómo Funciona LDA?

1. **Asignación Inicial**: LDA comienza asignando aleatoriamente cada palabra en cada documento a un tema.
2. **Iteración**: Luego, el algoritmo ajusta estas asignaciones iterativamente, basándose en la probabilidad de que una palabra pertenezca a un tema dado el contexto del documento.
3. **Convergencia**: Después de varias iteraciones, las asignaciones de temas se estabilizan, revelando los temas subyacentes en los documentos.

### Ejemplo Simplificado

Imagina que tienes tres documentos:

- Documento 1: "gato perro ratón"
- Documento 2: "perro ratón elefante"
- Documento 3: "gato elefante ratón"

LDA podría descubrir que hay dos temas principales:

- Tema 1: "gato", "perro", "ratón"
- Tema 2: "elefante", "ratón"

Y podría determinar que:

- Documento 1 es 100% Tema 1.
- Documento 2 es 50% Tema 1 y 50% Tema 2.
- Documento 3 es 100% Tema 2.

### Aplicaciones de LDA

LDA se utiliza en diversas aplicaciones, como:

- **Análisis de Contenido**: Descubrir temas en artículos de noticias, reseñas de productos o publicaciones en redes sociales.
- **Recomendación de Contenidos**: Sugerir artículos o productos basados en los temas de interés del usuario.
- **Investigación Académica**: Analizar grandes volúmenes de literatura científica para identificar tendencias y temas emergentes.

### Conclusión

LDA es una herramienta poderosa para el análisis de textos que permite descubrir temas ocultos en grandes colecciones de documentos. Su capacidad para identificar patrones y temas hace que sea una técnica valiosa en muchas áreas, desde el análisis de contenido hasta la recomendación de productos.

### Implementación de LDA con Gensim


#### 1. Introducción e Instalación

Gensim es una librería popular en Python para modelado semántico, facilitando el uso de técnicas como LDA y Word2Vec. Para instalarla, usa:


In [ ]:
# !pip install gensim


#### 2. Preprocesado y Creación del Diccionario

Primero, limpiamos y normalizamos los textos, transformándolos en listas de tokens. Luego, creamos un diccionario que asigna un identificador único a cada palabra.


In [ ]:
import nltk
import re
from gensim import corpora
from gensim.utils import simple_preprocess

nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('spanish'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[\W_]+', ' ', text)
    tokens = simple_preprocess(text, deacc=False)
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

documents = []
with open("text_mining.txt", "r", encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            documents.append(preprocess_text(line))

dictionary = corpora.Dictionary(documents)
print(f"Data dictionary: {dictionary.token2id}")


#### 3. Generar el Corpus (Bag-of-Words)

Convertimos cada documento en una lista de pares (word_id, count) usando `doc2bow()`.


In [ ]:
corpus_bow = [dictionary.doc2bow(doc) for doc in documents]
print(f"Bag of words: {corpus_bow}")

#### 4. Entrenamiento del Modelo LDA

Entrenamos el modelo LDA con el corpus y el diccionario, especificando el número de temas.


In [ ]:
from gensim.models.ldamodel import LdaModel

num_topics = 2
lda_model = LdaModel(
    corpus=corpus_bow,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    update_every=1,
    chunksize=100,
    alpha='auto',
    per_word_topics=True
)

for idx, topic in lda_model.show_topics(formatted=False, num_words=5):
    print(f"Tópico #{idx}: {[w for w, _ in topic]}")

display(lda_model.print_topics())

#### 5. Obtener la Distribución de Tópicos para un Documento

Usamos `.get_document_topics()` para obtener la distribución de tópicos en cada documento.


In [ ]:
for doc_idx, doc_bow in enumerate(corpus_bow):
    topics_distribution = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
    print(f"Document {doc_idx} topics distribution: {topics_distribution}")


### 6 Evaluación y Afinación del Modelo

#### 6.1 Métricas: Perplexity y Coherencia

Para evaluar un modelo LDA, utilizamos métricas como **perplexity** y **coherencia de tópicos**.

- **Perplexity** mide la capacidad del modelo para predecir nuevas palabras. Un valor menor indica mejor desempeño.
- **Coherencia de Tópicos** evalúa la consistencia de las palabras más importantes de cada tema. Un valor mayor indica temas más coherentes.


### Perplexity

**Perplexity** es una métrica que mide qué tan bien un modelo de lenguaje predice una muestra de texto. En el contexto de LDA, se utiliza para evaluar la calidad del modelo de temas.

- **Intuición**: Imagina que tienes un conjunto de documentos y un modelo LDA que ha identificado varios temas. La perplexity mide qué tan bien el modelo puede predecir las palabras en los documentos basándose en los temas identificados.
- **Valores**: Un valor de perplexity más bajo indica un mejor desempeño del modelo, ya que significa que el modelo es mejor para predecir las palabras en los documentos.
- **Cálculo**: Matemáticamente, la perplexity es la exponencial de la entropía cruzada promedio por palabra. En términos simples, mide la incertidumbre del modelo al predecir las palabras.

### Coherence

**Coherence** es una métrica que evalúa la calidad de los temas generados por el modelo LDA. Mide qué tan coherentes son las palabras dentro de cada tema.

- **Intuición**: Imagina que tienes un tema identificado por el modelo LDA, como "deportes". La coherencia mide qué tan bien se relacionan las palabras dentro de ese tema, como "fútbol", "baloncesto" y "equipo".
- **Valores**: Un valor de coherencia más alto indica que las palabras dentro de cada tema están más relacionadas entre sí, lo que hace que los temas sean más interpretables y útiles.
- **Cálculo**: Existen varias formas de calcular la coherencia, pero una de las más comunes es la **coherencia c_v**, que mide la similitud de las palabras dentro de un tema utilizando la probabilidad de co-ocurrencia de palabras.

In [ ]:
from gensim.models import CoherenceModel

coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=documents,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_score = coherence_model_lda.get_coherence()
print(f"Coherencia: {coherence_score}")


Para analizar la variación de la coherencia en función del número de tópicos:


In [ ]:
import matplotlib.pyplot as plt

k_values = range(1, 10)
coherence_scores = []

for k in k_values:
    lda_model = LdaModel(
        corpus=corpus_bow,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        update_every=1,
        chunksize=100,
        alpha='auto',
        per_word_topics=True
    )
    coherence_model_lda = CoherenceModel(
        model=lda_model,
        texts=documents,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence_score = coherence_model_lda.get_coherence()
    coherence_scores.append(coherence_score)

plt.figure(figsize=(10, 6))
plt.plot(k_values, coherence_scores, marker='o')
plt.xlabel("Número de Tópicos (K)")
plt.ylabel("Coherencia (c_v)")
plt.title("Variación de la Coherencia en función de K")
plt.xticks(k_values)
plt.grid(True)
plt.show()


Para calcular la perplexity:


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

k_values = range(1, 10)
perplexity_scores = []

for k in k_values:
    lda_model = LdaModel(
        corpus=corpus_bow,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10,
        update_every=1,
        chunksize=100,
        alpha='auto',
        per_word_topics=True
    )
    perplexity = lda_model.log_perplexity(corpus_bow)
    # print(f"Perplexity: {perplexity}")

    perplexity = np.exp(-perplexity)
    # print(f"Perplexity: {perplexity}")
    perplexity_scores.append(perplexity)

plt.figure(figsize=(10, 6))
plt.plot(k_values, perplexity_scores, marker='o')
plt.xlabel("Número de Tópicos (K)")
plt.ylabel("Perplexidad (c_v)")
plt.title("Variación de la Coherencia en función de K")
plt.xticks(k_values)
plt.grid(True)
plt.show()


#### 6.2 Visualización con pyLDAvis
Para explorar interactivamente los tópicos, usamos pyLDAvis:


In [ ]:
# !pip install pyLDAvis # Instalación

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

lda_model = LdaModel(
    corpus=corpus_bow,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=10,
    update_every=1,
    chunksize=100,
    alpha='auto',
    per_word_topics=True
)

pyLDAvis.enable_notebook()
visualization = gensimvis.prepare(lda_model, corpus_bow, dictionary)
pyLDAvis.display(visualization)

Los círculos representan la separación entre los temas. Si están superpuestos son temas semejantes, si están separados son diferentes.

Las barras azules representan la frecuencia de la palabra en general.
Las barras rojas representan las palabras que se pueden emplear para identificar los temas en los que aparecen en su respectiva importancia. Si aparecen barras rojas, son palabras que se pueden utilizar para identificar el tema.


## Enlaces

Documentación sobre una implementación en python: https://naturale0.github.io/2021/02/16/LDA-4-Gibbs-Sampling

Documentación sobre pyLDAvis : https://neptune.ai/blog/pyldavis-topic-modelling-exploration-tool-that-every-nlp-data-scientist-should-know